<a href="https://colab.research.google.com/github/UDAYNANNAKA/AI-resume-Analyzer/blob/main/resume_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# AI RESUME ANALYZER - COMPLETE PROJECT
# Google Colab - ONE CELL
# ============================================================

# Install required libraries
!pip -q install gradio pypdf scikit-learn pandas

import re
import os
import json
import pandas as pd
import gradio as gr

from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. SKILLS DATABASE
# ============================================================

SKILLS = [
    # Programming
    "python", "java", "javascript", "typescript", "c", "c++", "c#",
    "sql", "r", "go", "php",

    # Web
    "html", "css", "react", "angular", "vue", "node.js", "node",
    "express", "flask", "django",

    # Data
    "pandas", "numpy", "matplotlib", "seaborn", "excel",
    "power bi", "tableau", "data analysis", "data visualization",

    # Machine Learning / AI
    "machine learning", "deep learning", "artificial intelligence",
    "tensorflow", "pytorch", "scikit-learn", "nlp",
    "natural language processing", "computer vision",

    # Database
    "mysql", "postgresql", "mongodb", "oracle", "sqlite",

    # Cloud / DevOps
    "aws", "azure", "google cloud", "docker", "kubernetes",
    "git", "github", "gitlab", "linux",

    # Other
    "rest api", "api", "communication", "leadership",
    "problem solving", "teamwork", "agile", "scrum"
]


# ============================================================
# 2. EXTRACT TEXT FROM PDF
# ============================================================

def extract_pdf_text(file_path):

    if file_path is None:
        return ""

    try:
        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        return text

    except Exception as e:
        return f"ERROR: {str(e)}"


# ============================================================
# 3. CLEAN TEXT
# ============================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# 4. FIND SKILLS
# ============================================================

def find_skills(text):

    text = clean_text(text)

    found = []

    for skill in SKILLS:

        skill_clean = skill.lower()

        if skill_clean in text:
            found.append(skill)

    return sorted(list(set(found)))


# ============================================================
# 5. ATS SCORE
# ============================================================

def calculate_ats_score(resume_text, job_description):

    resume = clean_text(resume_text)
    job = clean_text(job_description)

    if not resume:
        return 0

    if not job:
        return 0

    # ----------------------------
    # Skill score
    # ----------------------------

    resume_skills = set(find_skills(resume))
    job_skills = set(find_skills(job))

    if job_skills:

        matched_skills = resume_skills.intersection(job_skills)

        skill_score = (
            len(matched_skills) / len(job_skills)
        ) * 60

    else:

        matched_skills = set()

        skill_score = 0


    # ----------------------------
    # TF-IDF similarity
    # ----------------------------

    try:

        vectorizer = TfidfVectorizer(
            stop_words="english"
        )

        vectors = vectorizer.fit_transform(
            [resume, job]
        )

        similarity = cosine_similarity(
            vectors[0:1],
            vectors[1:2]
        )[0][0]

        similarity_score = similarity * 30

    except:

        similarity_score = 0


    # ----------------------------
    # Resume structure score
    # ----------------------------

    sections = [
        "education",
        "experience",
        "skills",
        "project",
        "projects",
        "certification",
        "certifications"
    ]

    section_count = sum(
        1 for section in sections
        if section in resume
    )

    structure_score = min(
        section_count / 5,
        1
    ) * 10


    total = (
        skill_score +
        similarity_score +
        structure_score
    )

    return round(min(total, 100), 2)


# ============================================================
# 6. GENERATE ANALYSIS
# ============================================================

def analyze_resume(file, job_description):

    # ----------------------------
    # Validate input
    # ----------------------------

    if file is None:

        return (
            "❌ Please upload a PDF resume.",
            "",
            "",
            "",
            "",
            None
        )

    if not job_description.strip():

        return (
            "❌ Please enter a job description.",
            "",
            "",
            "",
            "",
            None
        )


    # ----------------------------
    # Extract resume
    # ----------------------------

    resume_text = extract_pdf_text(file)

    if not resume_text.strip():

        return (
            "❌ Could not extract text from the PDF.",
            "",
            "",
            "",
            "",
            None
        )


    # ----------------------------
    # Find skills
    # ----------------------------

    resume_skills = set(
        find_skills(resume_text)
    )

    job_skills = set(
        find_skills(job_description)
    )

    matched = sorted(
        resume_skills.intersection(job_skills)
    )

    missing = sorted(
        job_skills - resume_skills
    )


    # ----------------------------
    # Score
    # ----------------------------

    score = calculate_ats_score(
        resume_text,
        job_description
    )


    # ----------------------------
    # Recommendations
    # ----------------------------

    suggestions = []

    if score >= 80:

        suggestions.append(
            "Excellent match. Your resume is strongly aligned with the job."
        )

    elif score >= 60:

        suggestions.append(
            "Good match, but you can improve your resume by adding missing job-related skills."
        )

    elif score >= 40:

        suggestions.append(
            "Moderate match. Add relevant skills, projects and keywords from the job description."
        )

    else:

        suggestions.append(
            "Low match. Consider improving your skills section and adding relevant projects."
        )


    if missing:

        suggestions.append(
            "Add or develop these relevant skills: "
            + ", ".join(missing[:10])
        )


    if "projects" not in clean_text(resume_text):

        suggestions.append(
            "Add a Projects section with 2-3 practical projects."
        )


    if "certification" not in clean_text(resume_text):

        suggestions.append(
            "Consider adding relevant certifications."
        )


    if "github" not in clean_text(resume_text):

        suggestions.append(
            "Add your GitHub profile to your resume."
        )


    if "linkedin" not in clean_text(resume_text):

        suggestions.append(
            "Add your LinkedIn profile."
        )


    # ----------------------------
    # Result formatting
    # ----------------------------

    score_result = f"""
# 🎯 ATS RESUME SCORE

## {score}/100

### Score Interpretation

{"🟢 Excellent" if score >= 80 else
 "🟡 Good" if score >= 60 else
 "🟠 Needs Improvement" if score >= 40 else
 "🔴 Low Match"}
"""


    matched_result = "\n".join(
        f"✅ {skill.title()}"
        for skill in matched
    )

    if not matched_result:

        matched_result = "No matching skills detected."


    missing_result = "\n".join(
        f"❌ {skill.title()}"
        for skill in missing
    )

    if not missing_result:

        missing_result = "🎉 No major missing skills detected."


    suggestions_result = "\n".join(
        f"{i+1}. {s}"
        for i, s in enumerate(suggestions)
    )


    # ----------------------------
    # Create report
    # ----------------------------

    report = f"""
AI RESUME ANALYZER REPORT
=========================

ATS SCORE
---------
{score}/100

MATCHED SKILLS
--------------
{", ".join(matched) if matched else "None"}

MISSING SKILLS
--------------
{", ".join(missing) if missing else "None"}

RECOMMENDATIONS
---------------
{chr(10).join(suggestions)}

RESUME TEXT
-----------
{resume_text[:10000]}
"""


    report_file = "/content/resume_analysis_report.txt"

    with open(
        report_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(report)


    return (
        score_result,
        matched_result,
        missing_result,
        suggestions_result,
        resume_text[:5000],
        report_file
    )


# ============================================================
# 7. GRADIO INTERFACE
# ============================================================

css = """
body {
    font-family: Arial, sans-serif;
}

.gradio-container {
    max-width: 1100px !important;
}

h1 {
    text-align: center;
}
"""


with gr.Blocks(
    css=css,
    title="AI Resume Analyzer"
) as app:

    gr.Markdown(
        """
# 🤖 AI Resume Analyzer

### Analyze your resume against a job description

Upload your **PDF resume**, paste the **Job Description**, and get an ATS-style analysis.
"""
    )


    with gr.Row():

        with gr.Column():

            resume_file = gr.File(
                label="📄 Upload Resume (PDF)",
                file_types=[".pdf"],
                type="filepath"
            )

            job_description = gr.Textbox(
                label="💼 Job Description",
                placeholder="Paste the complete job description here...",
                lines=15
            )

            analyze_button = gr.Button(
                "🚀 Analyze Resume",
                variant="primary"
            )


        with gr.Column():

            score_output = gr.Markdown(
                label="ATS Score"
            )


            matched_output = gr.Markdown(
                label="Matching Skills"
            )


            missing_output = gr.Markdown(
                label="Missing Skills"
            )


            suggestions_output = gr.Markdown(
                label="Recommendations"
            )


    resume_text_output = gr.Textbox(
        label="📄 Extracted Resume Text",
        lines=15
    )


    report_output = gr.File(
        label="📥 Download Analysis Report"
    )


    analyze_button.click(
        fn=analyze_resume,
        inputs=[
            resume_file,
            job_description
        ],
        outputs=[
            score_output,
            matched_output,
            missing_output,
            suggestions_output,
            resume_text_output,
            report_output
        ]
    )


    gr.Markdown(
        """
---
### 🛠️ Technologies Used

**Python • PDF Processing • Scikit-learn • TF-IDF • Cosine Similarity • Gradio • NLP**

### 📌 Project for GitHub

This project demonstrates:

- Python programming
- NLP
- Machine Learning
- Data processing
- PDF extraction
- Web application development
- ATS resume analysis
"""
    )


# ============================================================
# 8. LAUNCH
# ============================================================

app.launch(
    share=True,
    debug=False
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 7.6 MB/s eta 0:00:00


/tmp/ipykernel_973/808190394.py:474: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://123aa06697bb6e9143.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
